# Loan Approval Prediction — Initial Exploration

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Load the dataset

In [3]:
DATA_PATH = "../data/loan_approval_dataset.csv"

df = pd.read_csv(DATA_PATH)

# The Kaggle version of this dataset has leading whitespace in column names
# and string values (e.g. " Approved" instead of "Approved") — strip both.
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print(f"Shape: {df.shape}")
df.head()


Shape: (4269, 13)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


## 2. Basic structure

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   loan_id                   4269 non-null   int64 
 1   no_of_dependents          4269 non-null   int64 
 2   education                 4269 non-null   object
 3   self_employed             4269 non-null   object
 4   income_annum              4269 non-null   int64 
 5   loan_amount               4269 non-null   int64 
 6   loan_term                 4269 non-null   int64 
 7   cibil_score               4269 non-null   int64 
 8   residential_assets_value  4269 non-null   int64 
 9   commercial_assets_value   4269 non-null   int64 
 10  luxury_assets_value       4269 non-null   int64 
 11  bank_asset_value          4269 non-null   int64 
 12  loan_status               4269 non-null   object
dtypes: int64(10), object(3)
memory usage: 433.7+ KB


In [5]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
loan_id,4269.0,NaN,NaN,NaN,2135.0,1232.498479,1.0,1068.0,2135.0,3202.0,4269.0
no_of_dependents,4269.0,NaN,NaN,NaN,2.498712,1.69591,0.0,1.0,3.0,4.0,5.0
education,4269,2,Graduate,2144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
self_employed,4269,2,Yes,2150,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_annum,4269.0,NaN,NaN,NaN,5059123.916608,2806839.831818,200000.0,2700000.0,5100000.0,7500000.0,9900000.0
loan_amount,4269.0,NaN,NaN,NaN,15133450.456781,9043362.984843,300000.0,7700000.0,14500000.0,21500000.0,39500000.0
loan_term,4269.0,NaN,NaN,NaN,10.900445,5.709187,2.0,6.0,10.0,16.0,20.0
cibil_score,4269.0,NaN,NaN,NaN,599.936051,172.430401,300.0,453.0,600.0,748.0,900.0
residential_assets_value,4269.0,NaN,NaN,NaN,7472616.537831,6503636.587664,-100000.0,2200000.0,5600000.0,11300000.0,29100000.0
commercial_assets_value,4269.0,NaN,NaN,NaN,4973155.305692,4388966.089638,0.0,1300000.0,3700000.0,7600000.0,19400000.0


## 3. Missing values

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).sort_values("missing_count", ascending=False)


,missing_count,missing_pct
loan_id,0,0.0
no_of_dependents,0,0.0
education,0,0.0
self_employed,0,0.0
income_annum,0,0.0
loan_amount,0,0.0
loan_term,0,0.0
cibil_score,0,0.0
residential_assets_value,0,0.0
commercial_assets_value,0,0.0


## 4. Duplicate rows

In [7]:
print(f"Duplicate rows: {df.duplicated().sum()}")


Duplicate rows: 0


## 5. Target variable balance

In [8]:
df["loan_status"].value_counts(normalize=True).round(3) * 100


loan_status
Approved    62.2
Rejected    37.8
Name: proportion, dtype: float64

## 6. Initial observations

1. Successfully loaded the loan approval dataset into a Pandas DataFrame and standardized the column names for consistency.

2. Explored the dataset structure by examining the number of records, features, data types, and memory usage to gain an overall understanding of the data.

3. Generated descriptive statistics for both numerical and categorical features to identify their distributions and basic characteristics.

4. Performed data quality checks by identifying missing values and duplicate records to determine the preprocessing requirements.

5. Analyzed the distribution of the target variable (`loan_status`) to understand the balance between approved and rejected loan applications.

6. The insights obtained from the initial exploration establish a clear understanding of the dataset and provide the foundation for subsequent preprocessing, feature engineering, exploratory data analysis, and predictive model development.

- **Shape:** 4,269 rows × 13 columns.
- **Missing values:** none — every column is 100% populated (0 missing across the board).
- **Duplicate rows:** none (`df.duplicated().sum()` = 0).
- **Target class balance:** moderately imbalanced — **62.2% Approved / 37.8% Rejected**. Not severe enough to require SMOTE/resampling, but worth using **F1 / ROC-AUC** rather than raw accuracy when comparing models in Day 4, and worth stratifying the train/test split on `loan_status`.
- **Numeric columns (10):** `loan_id`, `no_of_dependents`, `income_annum`, `loan_amount`, `loan_term`, `cibil_score`, `residential_assets_value`, `commercial_assets_value`, `luxury_assets_value`, `bank_asset_value`.
- **Categorical columns (2 features + 1 target):** `education` (Graduate / Not Graduate — near 50/50 split), `self_employed` (Yes / No — near 50/50 split), `loan_status` (target).

**Data quality flags to handle in further tasks**
- `residential_assets_value` has a **minimum of -100,000** — a negative asset value isn't physically meaningful. Needs investigation: likely a data entry artifact. Candidate fixes: clip to 0, treat as missing and impute, or drop the row if isolated.
- `loan_id` is a pure row identifier (1 to 4,269, no predictive signal) — drop before modeling to avoid leakage/noise.
- `cibil_score` ranges 300–900 with a wide std (~172), consistent with real-world CIBIL score range — no cleaning needed here, but worth a boxplot to check for outlier clusters near the boundaries.
- Asset value columns (`residential_assets_value`, `commercial_assets_value`, `luxury_assets_value`, `bank_asset_value`) have large scale differences and likely right-skewed distributions (means well above medians implied by the spread) — good candidates for scaling in  and log-transform checks.